Writing Python that works is the starting point. Writing Python that is readable, maintainable, and correct six months later — by you or a teammate — is the real goal. This notebook distils the habits, idioms, and tooling choices that professional Python developers rely on: following PEP 8, writing Pythonic code, handling errors well, using type hints effectively, structuring projects for growth, writing useful documentation, profiling before optimising, and applying the tooling that enforces these standards automatically.

## PEP 8 — The Style Guide

PEP 8 is Python's official style guide. Consistent formatting makes code easier to read at a glance, reduces cognitive load during reviews, and lets automated tools (formatters, linters) enforce rules so humans can focus on logic.

**Naming conventions:**

| What | Convention | Example |
|---|---|---|
| Variables, functions, methods | `snake_case` | `user_count`, `get_price()` |
| Classes | `PascalCase` | `ShoppingCart`, `HTTPClient` |
| Constants | `UPPER_SNAKE_CASE` | `MAX_RETRIES`, `DEFAULT_TIMEOUT` |
| Private attributes | `_single_leading_underscore` | `_cache`, `_validate()` |
| Name-mangled attributes | `__double_leading_underscore` | `__id` (use sparingly) |
| "Dunder" methods | `__double_both__` | `__init__`, `__repr__` |

**Formatting rules (the most important ones):**
- Indent with **4 spaces** (never tabs)
- Maximum line length **88 characters** (Black's default; PEP 8 says 79)
- Two blank lines around top-level definitions; one blank line between methods
- Imports at the top of the file, one per line, in three groups: stdlib → third-party → local
- No trailing whitespace; a single newline at end of file

In [ ]:
# --- Bad style ---
def calculateTotalPrice(ItemList,discount):
  total=0
  for i in ItemList:
    total=total+i['price']*i['qty']
  return total*(1-discount)

# --- Good style ---
MAX_DISCOUNT = 0.50

def calculate_total_price(items: list[dict], discount: float = 0.0) -> float:
    """Return the discounted total for a list of order items."""
    if not 0.0 <= discount <= MAX_DISCOUNT:
        raise ValueError(f"Discount must be between 0 and {MAX_DISCOUNT}")

    subtotal = sum(item["price"] * item["qty"] for item in items)
    return subtotal * (1 - discount)


items = [{"price": 10.0, "qty": 3}, {"price": 5.0, "qty": 2}]
print(calculate_total_price(items, discount=0.1))

In [ ]:
# Import ordering — stdlib, third-party, local; one import per line

# --- Bad ---
# import os, sys
# from collections import Counter, defaultdict, OrderedDict
# import numpy, pandas
# from myapp.models import User

# --- Good ---
# Standard library
import os
import sys
from collections import Counter, defaultdict

# Third-party (installed via pip)
# import numpy as np
# import pandas as pd

# Local application
# from myapp.models import User

# isort and ruff enforce this ordering automatically

# Boolean comparisons — never use == with True/False/None
flag = True
value = None

# Bad
if flag == True:  pass
if value == None: pass

# Good
if flag:          pass
if value is None: pass   # 'is None' for identity check

# Checking emptiness — use truthiness, not len()
items = []
if not items:                  print("empty — good")
if len(items) == 0:            print("empty — bad")   # noqa

## Pythonic Idioms

"Pythonic" means using the language's built-in features and conventions the way they were designed — not translating patterns from Java or C. Pythonic code is shorter, clearer, and often faster.

In [ ]:
# EAFP (Easier to Ask Forgiveness than Permission)
# Python style: try the operation, handle the exception if it fails
# vs LBYL (Look Before You Leap): guard with if-checks first

data = {"user": {"name": "Alice", "age": 30}}

# LBYL — defensive checks before every operation
if "user" in data and "name" in data["user"]:
    name = data["user"]["name"]
else:
    name = "unknown"

# EAFP — just try it
try:
    name = data["user"]["name"]
except (KeyError, TypeError):
    name = "unknown"

# For simple cases, .get() is cleaner than both
name = data.get("user", {}).get("name", "unknown")
print(name)

# Unpacking — exchange, multi-assign, ignore parts with _
a, b = 1, 2
a, b = b, a                  # swap without temp variable
print(a, b)                  # 2 1

first, *rest = [1, 2, 3, 4, 5]
print(first, rest)           # 1 [2, 3, 4, 5]

*head, last = [1, 2, 3, 4, 5]
print(head, last)            # [1, 2, 3, 4] 5

first, *_, last = [1, 2, 3, 4, 5]
print(first, last)           # 1 5

# Unpack in for loops
pairs = [("Alice", 95), ("Bob", 87), ("Carol", 92)]
for name, score in pairs:
    print(f"{name}: {score}")

In [ ]:
# enumerate and zip — prefer these over manual index tracking
fruits = ["apple", "banana", "cherry"]

# Bad — manual index
for i in range(len(fruits)):
    print(i, fruits[i])

# Good — enumerate
for i, fruit in enumerate(fruits, start=1):
    print(f"{i}. {fruit}")

# zip — iterate over multiple sequences in parallel
names  = ["Alice", "Bob", "Carol"]
scores = [95, 87, 92]
grades = ["A", "B", "A"]

# Bad
for i in range(len(names)):
    print(names[i], scores[i], grades[i])

# Good
for name, score, grade in zip(names, scores, grades):
    print(f"{name}: {score} ({grade})")

# zip_longest — pair sequences of different lengths, filling missing values
from itertools import zip_longest
short = [1, 2, 3]
long  = ["a", "b", "c", "d", "e"]
for n, c in zip_longest(short, long, fillvalue=0):
    print(n, c)

In [ ]:
# Walrus operator (:=) — assign inside an expression (Python 3.8+)
import re

text = "Order #12345 placed on 2024-06-15"

# Without walrus — match() called twice or split across two lines
m = re.search(r"#(\d+)", text)
if m:
    print(f"Order number: {m.group(1)}")

# With walrus — assign and test in one step
if m := re.search(r"#(\d+)", text):
    print(f"Order number: {m.group(1)}")

# Useful in while loops — read until sentinel
import io
stream = io.StringIO("line1\nline2\nline3")
while line := stream.readline():
    print(line.strip())

# Comprehension with a condition that reuses a computed value
data = ["alice@example.com", "not-an-email", "bob@test.org", ""]
EMAIL_RE = re.compile(r"^[\w.+-]+@[\w-]+\.[\w.]+$")
valid_emails = [m.group() for item in data if (m := EMAIL_RE.match(item))]
print(valid_emails)

In [ ]:
# Use built-ins instead of reinventing them
numbers = [3, 1, 4, 1, 5, 9, 2, 6]

# Prefer
print(min(numbers), max(numbers), sum(numbers))
print(sorted(numbers))
print(any(n > 8 for n in numbers))    # True if any matches
print(all(n > 0 for n in numbers))    # True if all match

# sorted with key — sort by derived value without changing the original
words = ["banana", "apple", "cherry", "date"]
print(sorted(words, key=len))                       # by length
print(sorted(words, key=str.lower))                 # case-insensitive
print(sorted(words, key=len, reverse=True))         # longest first

# Prefer dict.get over try/except for simple defaults
config = {"timeout": 30}
retries = config.get("retries", 3)    # default 3 if key absent
print(retries)

# setdefault — add key only if absent, return the value
cache = {}
cache.setdefault("hits", 0)
cache["hits"] += 1
print(cache)

# dict | merge operator (Python 3.9+)
defaults = {"debug": False, "timeout": 30}
overrides = {"timeout": 60, "verbose": True}
merged = defaults | overrides      # overrides win
print(merged)

## Error Handling

Good error handling communicates what went wrong, to whom, and what they can do about it. It distinguishes between errors you can recover from and errors that indicate a programming bug.

In [ ]:
# Catch the most specific exception possible
import json

def load_config(path: str) -> dict:
    # --- Bad: bare except swallows everything including KeyboardInterrupt ---
    # try:
    #     with open(path) as f:
    #         return json.load(f)
    # except:
    #     return {}

    # --- Also bad: Exception is too broad for meaningful recovery ---
    # except Exception:
    #     return {}

    # --- Good: specific exceptions with specific responses ---
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError:
        return {}                         # config is optional — empty dict is fine
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid config file {path!r}: {e}") from e

print(load_config("/nonexistent/config.json"))   # {}

# Exception chaining — preserve the original cause with 'raise ... from e'
class DatabaseError(Exception): pass

def fetch_user(user_id: int) -> dict:
    try:
        # Simulate a low-level error
        raise ConnectionError("socket timeout")
    except ConnectionError as e:
        raise DatabaseError(f"Could not fetch user {user_id}") from e

try:
    fetch_user(42)
except DatabaseError as e:
    print(f"Error: {e}")
    print(f"Caused by: {e.__cause__}")

In [ ]:
# Custom exception hierarchies
class AppError(Exception):
    """Base for all application errors — catch this to catch any app error."""

class ValidationError(AppError):
    """Input failed validation."""
    def __init__(self, field: str, message: str):
        self.field = field
        self.message = message
        super().__init__(f"{field}: {message}")

class NotFoundError(AppError):
    """Requested resource does not exist."""
    def __init__(self, resource: str, identifier):
        self.resource = resource
        self.identifier = identifier
        super().__init__(f"{resource} {identifier!r} not found")

class PermissionError_(AppError):   # renamed to avoid shadowing built-in
    """Caller lacks permission to perform the action."""


def get_order(order_id: int, user_id: int) -> dict:
    orders = {1: {"id": 1, "user_id": 42, "total": 99.0}}
    if order_id not in orders:
        raise NotFoundError("Order", order_id)
    order = orders[order_id]
    if order["user_id"] != user_id:
        raise PermissionError_("Access denied")
    return order

for oid, uid in [(1, 42), (1, 99), (99, 42)]:
    try:
        print(get_order(oid, uid))
    except AppError as e:
        print(f"{type(e).__name__}: {e}")

In [ ]:
# else and finally in try blocks
def read_integer(prompt: str) -> int:
    """Demonstrate try/except/else/finally."""
    value = "42"   # simulated input
    try:
        result = int(value)
    except ValueError:
        print("Not a valid integer")
        return 0
    else:
        # runs only if no exception was raised in try
        print(f"Parsed successfully: {result}")
        return result
    finally:
        # always runs — good for cleanup that must happen regardless
        print("(finally block ran)")

read_integer("Enter a number: ")

# Don't use exceptions for normal flow control
# --- Bad ---
def get_first_bad(items):
    try:
        return items[0]
    except IndexError:
        return None

# --- Good ---
def get_first_good(items):
    return items[0] if items else None

print(get_first_bad([]),  get_first_bad([1, 2, 3]))
print(get_first_good([]), get_first_good([1, 2, 3]))

## Type Hints in Practice

Type hints (covered in depth in notebook 16) improve readability, enable IDE autocompletion, and allow static analysis with mypy or pyright to catch bugs before runtime. The goal is hints that add value without becoming noise.

In [ ]:
from __future__ import annotations   # makes all annotations strings — deferred evaluation
from typing import TypeVar, Generic, Protocol, overload, Literal
from collections.abc import Sequence, Mapping, Iterator, Callable

# Use abstract types in function signatures — accept more, restrict less
# Sequence instead of list; Mapping instead of dict
def mean(values: Sequence[float]) -> float:
    """Works with lists, tuples, any sequence — not just list."""
    return sum(values) / len(values)

print(mean([1, 2, 3]))      # list
print(mean((1, 2, 3)))      # tuple — works because we typed Sequence

# TypeVar — generic functions
T = TypeVar("T")

def first(seq: Sequence[T]) -> T | None:
    return seq[0] if seq else None

# Protocols — structural subtyping ("duck typing with types")
class Closeable(Protocol):
    def close(self) -> None: ...

def close_all(resources: list[Closeable]) -> None:
    for r in resources:
        r.close()

# Anything with a .close() method satisfies Closeable — no inheritance required

# Literal — restrict to specific values
def set_log_level(level: Literal["DEBUG", "INFO", "WARNING", "ERROR"]) -> None:
    print(f"Level set to {level}")

set_log_level("INFO")
# set_log_level("VERBOSE")  # mypy would catch this

In [ ]:
from typing import TypedDict, NamedTuple

# TypedDict — type-check dict structures without a full class
class OrderItem(TypedDict):
    product_id: int
    name: str
    price: float
    qty: int

class Order(TypedDict):
    order_id: int
    items: list[OrderItem]
    discount: float

def process_order(order: Order) -> float:
    subtotal = sum(item["price"] * item["qty"] for item in order["items"])
    return subtotal * (1 - order["discount"])

order: Order = {
    "order_id": 1,
    "items": [{"product_id": 10, "name": "Widget", "price": 9.99, "qty": 3}],
    "discount": 0.1,
}
print(process_order(order))

# NamedTuple with types — like namedtuple but with annotation syntax
class Point(NamedTuple):
    x: float
    y: float
    label: str = ""    # optional with default

p = Point(1.0, 2.0, "origin")
print(p, p.x, p.label)

## Writing Good Docstrings

A docstring is a string literal that appears as the first statement in a module, class, or function. It becomes the object's `__doc__` attribute, is shown by `help()`, and is extracted by documentation generators.

Use **Google style** for its readability in source form, or **NumPy style** for scientific libraries. The key rule: document the contract (what the function does, what it expects, what it returns, what it raises) — not the implementation.

In [ ]:
def divide(numerator: float, denominator: float) -> float:
    """Divide numerator by denominator and return the quotient.

    Args:
        numerator: The number to be divided.
        denominator: The number to divide by. Must not be zero.

    Returns:
        The quotient as a float.

    Raises:
        ValueError: If denominator is zero.

    Examples:
        >>> divide(10, 2)
        5.0
        >>> divide(7, 2)
        3.5
    """
    if denominator == 0:
        raise ValueError("denominator must not be zero")
    return numerator / denominator


class EmailSender:
    """Send transactional email via an SMTP server.

    Attributes:
        host: SMTP server hostname.
        port: SMTP server port (default 587 for STARTTLS).

    Example:
        >>> sender = EmailSender("smtp.example.com", 587)
        >>> sender.send("alice@example.com", "Hi", "Hello Alice")
    """

    def __init__(self, host: str, port: int = 587) -> None:
        self.host = host
        self.port = port

    def send(self, to: str, subject: str, body: str) -> bool:
        """Send a plain-text email.

        Args:
            to: Recipient email address.
            subject: Email subject line.
            body: Plain-text body content.

        Returns:
            True if the email was accepted by the server.

        Raises:
            ConnectionError: If the SMTP server is unreachable.
        """
        print(f"[{self.host}:{self.port}] → {to}: {subject}")
        return True


# doctest — runnable examples embedded in docstrings
import doctest
doctest.run_docstring_examples(divide, globs={"divide": divide}, verbose=True)

help(divide)

## Project Structure

A well-structured project is navigable, testable, and deployable without confusion. The **`src` layout** is the recommended pattern for installable packages — it forces you to install the package to use it, which catches packaging bugs early.

```
my-project/
├── src/
│   └── mypackage/
│       ├── __init__.py
│       ├── core.py
│       └── utils.py
├── tests/
│   ├── conftest.py
│   ├── test_core.py
│   └── test_utils.py
├── pyproject.toml       ← single config file for everything
├── README.md
└── .gitignore
```

**`pyproject.toml`** is the modern single-file replacement for `setup.py`, `setup.cfg`, `requirements.txt`, `pytest.ini`, `.flake8`, and `mypy.ini`:

```toml
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "mypackage"
version = "0.1.0"
requires-python = ">=3.11"
dependencies = ["requests>=2.31", "pydantic>=2.0"]

[project.optional-dependencies]
dev = ["pytest", "pytest-cov", "ruff", "mypy"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts   = "-v --cov=src --cov-report=term-missing"

[tool.ruff.lint]
select = ["E", "F", "I", "UP"]

[tool.mypy]
strict = true
```

**Virtual environments** isolate each project's dependencies. Every project should have its own:

```bash
python -m venv .venv
source .venv/bin/activate    # macOS/Linux
.venv\Scripts\activate       # Windows
pip install -e ".[dev]"      # install in editable mode with dev extras
```

## Performance — Profile First, Then Optimise

The first rule of optimisation is **measure before you change anything**. Intuitions about bottlenecks are frequently wrong. Python provides profiling tools that tell you exactly where time is spent.

In [ ]:
import timeit
import cProfile
import pstats
import io

# timeit — measure small code snippets
# List comprehension vs map+list vs for loop
setup = "data = list(range(10_000))"

t_comp  = timeit.timeit("[x*x for x in data]",     setup=setup, number=1000)
t_map   = timeit.timeit("list(map(lambda x:x*x, data))", setup=setup, number=1000)
t_loop  = timeit.timeit(
    "result = []\nfor x in data: result.append(x*x)",
    setup=setup, number=1000
)

print(f"Comprehension: {t_comp:.3f}s")
print(f"map+lambda:    {t_map:.3f}s")
print(f"for loop:      {t_loop:.3f}s")

# cProfile — full call-graph profiling
def slow_function():
    return sum(i * i for i in range(100_000))

def main():
    for _ in range(5):
        slow_function()

pr = cProfile.Profile()
pr.enable()
main()
pr.disable()

s = io.StringIO()
ps = pstats.Stats(pr, stream=s).sort_stats("cumulative")
ps.print_stats(5)   # top 5 functions by cumulative time
print(s.getvalue())

In [ ]:
import sys

# Generators vs lists — memory efficiency for large sequences
# A generator does not build the full sequence in memory

def squares_list(n):
    return [i * i for i in range(n)]   # builds entire list in memory

def squares_gen(n):
    return (i * i for i in range(n))   # lazy — one value at a time

n = 1_000_000
lst = squares_list(n)
gen = squares_gen(n)

print(f"List size:      {sys.getsizeof(lst):,} bytes")
print(f"Generator size: {sys.getsizeof(gen):,} bytes")

# Both support sum() — generator is equally convenient but far more memory-efficient
print(sum(squares_list(1000)) == sum(squares_gen(1000)))

In [ ]:
# __slots__ — reduce per-instance memory for classes with fixed attributes
class PointRegular:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class PointSlots:
    __slots__ = ("x", "y")   # disables instance __dict__; only these attributes allowed
    def __init__(self, x, y):
        self.x = x
        self.y = y

regular = PointRegular(1, 2)
slotted = PointSlots(1, 2)

print(f"Regular: {sys.getsizeof(regular)} bytes + dict {sys.getsizeof(regular.__dict__)} bytes")
print(f"Slots:   {sys.getsizeof(slotted)} bytes (no __dict__)")

# When to use __slots__:
# - You create millions of instances of the same class
# - The class has a fixed set of attributes that will not change
# Trade-off: can't add arbitrary attributes; can't use weakref without adding it to __slots__

# String concatenation — use join() or f-strings, not repeated +
parts = [f"item_{i}" for i in range(1000)]

# Bad — creates a new string object on every iteration: O(n²)
# result = ""
# for p in parts:
#     result += p + ", "

# Good — join allocates once
result = ", ".join(parts)
print(result[:50])

## Security Basics

Security vulnerabilities often come from trusting user-supplied input too much, or from using convenient but unsafe functions. A few rules eliminate the most common pitfalls.

In [ ]:
import secrets
import hashlib
import hmac

# Never use eval() or exec() on user-supplied input
# eval() executes arbitrary Python — a classic code injection vector

# Bad
# user_input = input("Enter expression: ")
# result = eval(user_input)  # user can type: __import__('os').system('rm -rf /')

# Good — parse the specific structure you need
import ast

def safe_eval_int(expr: str) -> int:
    """Evaluate a simple integer arithmetic expression safely."""
    tree = ast.parse(expr, mode="eval")
    # Only allow literal numbers and basic arithmetic ops
    allowed = (ast.Expression, ast.BinOp, ast.UnaryOp,
               ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Constant)
    for node in ast.walk(tree):
        if not isinstance(node, allowed):
            raise ValueError(f"Disallowed expression: {type(node).__name__}")
    return eval(compile(tree, "<string>", "eval"))  # noqa: S307 — controlled input

print(safe_eval_int("2 + 3 * 4"))  # 14

try:
    safe_eval_int("__import__('os').getcwd()")
except ValueError as e:
    print(f"Blocked: {e}")

In [ ]:
# SQL injection — always use parameterised queries, never string formatting
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (id INTEGER, name TEXT, email TEXT)")
conn.execute("INSERT INTO users VALUES (1, 'Alice', 'alice@example.com')")
conn.execute("INSERT INTO users VALUES (2, 'Bob',   'bob@example.com')")
conn.commit()

user_input = "' OR '1'='1"   # classic SQL injection payload

# Bad — formats user input directly into SQL
# query = f"SELECT * FROM users WHERE name = '{user_input}'"  # returns ALL rows!
# conn.execute(query)

# Good — parameterised query; the db driver escapes the value
rows = conn.execute("SELECT * FROM users WHERE name = ?", (user_input,)).fetchall()
print(f"Parameterised result: {rows}")   # [] — no rows match the literal string

# Path traversal — validate that a path stays within the intended directory
from pathlib import Path

BASE_DIR = Path("/var/www/uploads")

def safe_path(filename: str) -> Path:
    """Return an absolute path inside BASE_DIR, raising if traversal is attempted."""
    target = (BASE_DIR / filename).resolve()
    if not target.is_relative_to(BASE_DIR):
        raise ValueError(f"Path traversal attempt: {filename!r}")
    return target

try:
    safe_path("../../../etc/passwd")
except ValueError as e:
    print(e)

In [ ]:
# Secrets and tokens — use secrets, not random
# random is seeded from the clock and is predictable

# Bad
import random as _random
token_bad = "".join(_random.choices("abcdef0123456789", k=32))

# Good — cryptographically secure
token_good = secrets.token_hex(16)     # 32-character hex token
reset_link = secrets.token_urlsafe(32)  # URL-safe token for password reset
print(token_good)
print(reset_link)

# Comparing secrets — use hmac.compare_digest to prevent timing attacks
# Regular == exits early on the first mismatch, leaking timing information
def verify_token(received: str, expected: str) -> bool:
    # hmac.compare_digest is constant-time
    return hmac.compare_digest(received.encode(), expected.encode())

stored = secrets.token_hex(16)
print(verify_token(stored, stored))    # True
print(verify_token("wrong", stored))   # False

# Hashing passwords — use hashlib.scrypt or a library like bcrypt/argon2
# Never store plain-text passwords
# Never use MD5 or SHA-1 for passwords — they are too fast
salt = secrets.token_bytes(16)
password_hash = hashlib.scrypt(b"user_password", salt=salt, n=2**14, r=8, p=1)
print(f"Hash length: {len(password_hash)} bytes")

## Tooling — Automate Code Quality

The best style guide is one enforced by tools, not argued about in code review. The modern Python toolchain is fast, opinionated, and integrates with every major editor and CI system.

| Tool | Purpose | Command |
|---|---|---|
| **ruff** | Linting + import sorting (replaces flake8, isort) | `ruff check .` |
| **ruff format** | Code formatting (replaces Black) | `ruff format .` |
| **mypy** | Static type checking | `mypy src/` |
| **pytest** | Test runner with fixtures and plugins | `pytest` |
| **pytest-cov** | Coverage measurement | `pytest --cov=src` |
| **pre-commit** | Run checks automatically before every commit | `pre-commit run --all-files` |

**`pre-commit` configuration (`.pre-commit-config.yaml`):**

```yaml
repos:
  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.4.0
    hooks:
      - id: ruff
        args: [--fix]
      - id: ruff-format

  - repo: https://github.com/pre-commit/mirrors-mypy
    rev: v1.9.0
    hooks:
      - id: mypy
        additional_dependencies: [types-requests]

  - repo: https://github.com/pre-commit/pre-commit-hooks
    rev: v4.6.0
    hooks:
      - id: trailing-whitespace
      - id: end-of-file-fixer
      - id: check-merge-conflict
      - id: check-json
      - id: check-yaml
```

Install once per repo: `pre-commit install`. After that, checks run on every `git commit` automatically.

## Design Principles

Good code is not just syntactically correct — it is designed to be understood, extended, and deleted.

In [ ]:
# Single Responsibility — each function does one thing

# Bad — one function does three jobs
def process_and_save_user(raw_data: dict) -> None:
    # validate
    if not raw_data.get("email"):
        raise ValueError("email required")
    # transform
    user = {"email": raw_data["email"].lower().strip(), "active": True}
    # persist (side effect)
    print(f"Saving {user}")   # imagine a DB write

# Good — three separate, testable functions
def validate_user_data(data: dict) -> None:
    if not data.get("email"):
        raise ValueError("email required")

def normalise_user(data: dict) -> dict:
    return {"email": data["email"].lower().strip(), "active": True}

def save_user(user: dict) -> None:
    print(f"Saving {user}")

def create_user(raw_data: dict) -> None:
    validate_user_data(raw_data)
    user = normalise_user(raw_data)
    save_user(user)

create_user({"email": "  Alice@Example.COM  "})

In [ ]:
# DRY (Don't Repeat Yourself) — but only when the abstraction is genuinely shared
# The wrong kind of DRY: two things that look similar but have different reasons to change

# --- Over-DRY (wrong abstraction) ---
def format_value(value, context):
    """Shared formatter that knows about too many contexts."""
    if context == "invoice":
        return f"${value:.2f}"
    elif context == "report":
        return f"{value:,.0f}"
    elif context == "api":
        return str(round(value, 4))
    # Every new context means editing this function

# --- Right approach: separate functions for separate contexts ---
def format_invoice_amount(value: float) -> str:
    return f"${value:.2f}"

def format_report_number(value: float) -> str:
    return f"{value:,.0f}"

def format_api_value(value: float) -> str:
    return str(round(value, 4))

# YAGNI (You Aren't Gonna Need It) — don't build for hypothetical future requirements
# If the requirement doesn't exist today, don't add a plugin system, config flag,
# or abstraction layer for it. Add it when the requirement arrives.

print(format_invoice_amount(1234.5))
print(format_report_number(1234567.89))
print(format_api_value(3.14159265))

In [ ]:
# Dependency injection — pass dependencies in; don't hardcode them
# Makes code testable and flexible without changing internals

from typing import Protocol

class EmailService(Protocol):
    def send(self, to: str, subject: str, body: str) -> bool: ...

# Bad — hardcodes the email implementation
class OrderServiceBad:
    def __init__(self):
        self._mailer = EmailSender("smtp.prod.example.com")   # can't swap in tests

    def place_order(self, order: dict) -> None:
        self._mailer.send(order["email"], "Order Confirmed", "Thanks!")

# Good — accepts any object that satisfies EmailService protocol
class OrderService:
    def __init__(self, mailer: EmailService) -> None:
        self._mailer = mailer   # injected — callers decide which implementation

    def place_order(self, order: dict) -> None:
        self._mailer.send(order["email"], "Order Confirmed", "Thanks for your order!")

# In tests, inject a fake
class FakeMailer:
    def __init__(self):
        self.sent: list[tuple] = []

    def send(self, to: str, subject: str, body: str) -> bool:
        self.sent.append((to, subject, body))
        return True

mailer = FakeMailer()
service = OrderService(mailer)
service.place_order({"email": "alice@example.com", "total": 49.99})
print(mailer.sent)   # [('alice@example.com', 'Order Confirmed', 'Thanks for your order!')]

# In production, inject the real one
# service = OrderService(EmailSender("smtp.example.com"))

In [ ]:
# Immutability — prefer immutable data where possible
# Mutable state is the source of many hard-to-find bugs

from dataclasses import dataclass, replace

@dataclass(frozen=True)
class Money:
    amount: float
    currency: str

    def add(self, other: "Money") -> "Money":
        if self.currency != other.currency:
            raise ValueError("Cannot add different currencies")
        return replace(self, amount=self.amount + other.amount)

    def __str__(self) -> str:
        return f"{self.amount:.2f} {self.currency}"

price    = Money(9.99, "USD")
shipping = Money(4.99, "USD")
total    = price.add(shipping)   # returns a new object — originals unchanged
print(price, shipping, total)

# Use tuples for fixed-size collections of different types
# Use lists for variable-size collections of the same type
coordinates = (48.8566, 2.3522)   # lat/lon pair — tuple is semantically correct
items = ["apple", "banana"]       # growing list — list is correct

# Use frozenset when you need an immutable set
ALLOWED_EXTENSIONS = frozenset({".jpg", ".jpeg", ".png", ".gif", ".webp"})
print(".jpg" in ALLOWED_EXTENSIONS)    # True
print(".exe" in ALLOWED_EXTENSIONS)    # False

## Summary

- **PEP 8** defines Python's style conventions: `snake_case` for functions and variables, `PascalCase` for classes, `UPPER_SNAKE_CASE` for constants. Indent with 4 spaces. Group imports (stdlib → third-party → local). Use `ruff format` to automate formatting and stop debating it in code review.

- **Pythonic idioms** make code shorter and clearer: EAFP (try first, handle exceptions) over LBYL (check before acting); starred unpacking for flexible assignment; `enumerate` and `zip` instead of manual index tracking; the walrus operator `:=` to assign inside conditions; built-ins (`min`, `max`, `sum`, `any`, `all`, `sorted` with `key=`) instead of manual loops.

- **Error handling**: catch specific exception types, not bare `except` or `except Exception`. Use `raise ... from e` to chain exceptions and preserve the original cause. Build a custom exception hierarchy rooted in a project-specific base class. Use `try/except/else/finally` — `else` runs only on success, `finally` always runs.

- **Type hints**: use abstract types (`Sequence`, `Mapping`, `Callable`) in function signatures to accept more. `TypedDict` documents dict shapes. `Protocol` provides structural typing. `Literal` restricts to specific values. Run `mypy --strict` in CI to catch type errors before they reach production.

- **Docstrings**: document the contract, not the implementation. Google style: `Args`, `Returns`, `Raises` sections with clear descriptions. Embed runnable examples with `>>>` for doctest.

- **Project structure**: use the `src` layout with `pyproject.toml` as the single configuration file. Keep tests alongside source in a `tests/` directory with a `conftest.py` for shared fixtures. Every project gets its own virtual environment.

- **Performance**: measure first with `timeit` and `cProfile`; optimise only where the data says. Prefer generators over lists for large sequences — constant memory vs linear. Use `__slots__` for classes instantiated millions of times. Join strings with `str.join`, not repeated `+`.

- **Security**: never pass user input to `eval` or `exec`. Always use parameterised queries — never format SQL by hand. Validate file paths against a base directory to prevent traversal. Use `secrets` for tokens and passwords, never `random`. Compare secret strings with `hmac.compare_digest` to prevent timing attacks. Hash passwords with a slow algorithm (`scrypt`, `bcrypt`, `argon2`).

- **Tooling**: `ruff` for linting and formatting, `mypy` for type checking, `pytest` with `pytest-cov` for testing, `pre-commit` to run all checks before every commit.

- **Design**: each function should have a single clear responsibility. Avoid premature abstraction — three similar lines are better than a wrong abstraction. Do not build for hypothetical future requirements (YAGNI). Inject dependencies rather than hardcoding them — this makes code testable without mocking internals. Prefer immutable data structures; mutable shared state is the root of many subtle bugs.